## LESSON 1: ENSEMBLE LEARNING

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
datas={
    "possession":[60, 45, 55, 70, 25, 40,35],
    "shots":[15, 7, 12, 13, 6, 17, 22],
    "shots_on_target":[8, 2, 6, 1, 5, 10, 9],
    "pass_accuracy":[75, 32, 82, 34, 25, 57, 49],
    "win":[1, 0, 1, 0, 0, 1, 1]
}
df=pd.DataFrame(datas)
X=df.drop("win", axis=1)
y=df["win"]
print(X.head())

X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models={
    "logistic_regression: ": LogisticRegression(max_iter=1000),
    "decision_tree_classifier: ":DecisionTreeClassifier(random_state=42),
    "random_forest_classifier :":RandomForestClassifier(n_estimators=100, random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    score=model.score(X_test, y_test)
    print(name, score)

ensemble=VotingClassifier(
    estimators=[
        ("dt", DecisionTreeClassifier(random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("lr", LogisticRegression(max_iter=1000))
    ],
    voting="hard"
)
ensemble.fit(X_train, y_train)
y_pred=ensemble.predict(X_test)
print("Ensemble Accuracy: ", ensemble.score(X_test, y_test))
print("Prediction", y_pred)

   possession  shots  shots_on_target  pass_accuracy
0          60     15                8             75
1          45      7                2             32
2          55     12                6             82
3          70     13                1             34
4          25      6                5             25
logistic_regression:  1.0
decision_tree_classifier:  0.5
random_forest_classifier : 1.0
Ensemble Accuracy:  1.0
Prediction [1 0]


# NOTE:
---
~~~ 
Ensemble Learning combines multiple models.
Bagging trains models independently using different samples.
Boosting trains models sequentially to improve predictions.
Stacking combines model predictions using another model.
Random Forest is an example of Bagging.
XGBoost and LightGBM are example of Boosting.
Voting combines predictions using majority voting or probability averaging.
~~~
---

## LESSON 2: Bagging Vs Boosting

In [1]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y=make_classification(
    n_samples=100,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=0.8,
    flip_y=0.05,
    random_state=42
)
feature_names=[
    "possession",
    "shots",
    "shots_on_target",
    "pass_accuracy",
    "corners",
    "tackles",
    "fouls",
    "distance_covered"
]
X=pd.DataFrame(X, columns=feature_names)
y=pd.Series(y, name="win")

X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Bagging implementation
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
bagging_model=BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
bagging_model.fit(X_train, y_train)
bagging_pred= bagging_model.predict(X_test)

# Boosting model
from sklearn.ensemble import AdaBoostClassifier
boosting_model=AdaBoostClassifier(
    estimator=DecisionTreeClassifier(),
    learning_rate=0.5,
    n_estimators=100,
    random_state=42
)
boosting_model.fit(X_train, y_train)
boosting_pred= boosting_model.predict(X_test)

# Comparision
from sklearn.metrics import accuracy_score, f1_score
results=pd.DataFrame({
    "Model":["Bagging", "Boosting(ADABoost)"],
    "Accuracy":[
        accuracy_score(y_test, bagging_pred),
        accuracy_score(y_test, boosting_pred)
    ],
    "F1_score":[
        f1_score(y_test, bagging_pred),
        f1_score(y_test, boosting_pred)
    ]
})

print(results)

                Model  Accuracy  F1_score
0             Bagging      0.85  0.823529
1  Boosting(ADABoost)      0.80  0.777778


# NOTE:
---
~~~ 
Bagging: Independent model training.
Bootstrap: Sampling with replacement.
Variance: Sensitivity to traning data.
Boosting: Sequential model improvement.
AdaBoost: Focuses on misclassified samples.
GradientBoosting: Learns Corrections using Gradients.
RandomForest: Bagging-based ensemble.
XGBoost: Advanced/Optimized Gradient Boosting
LightGBM: Efficient Gradient Boosting Framework.
~~~
---